# Module 4 — Reference Data from Papers + Multi-Dataset Integration + Motif Analysis

**Paired with:** Sessions 8–9 (paired-end mapping, Biopython, MEME)

**Lab context:** *E. coli* K-12 | bowtie2 | MEME Suite | Biopython | FUR transcription factor

---

This module covers three skills:

1. **Extracting structured data from papers using Claude** — binding sites, gene lists, and coordinates are often buried in supplementary tables or result paragraphs; Claude can pull them into a structured format (CSV/TSV).
2. **Paired-end alignment** — RNA-seq and ChIP-seq data often arrive as read pairs; understanding the key bowtie2 flags for paired-end mode is essential.
3. **Motif discovery with MEME** — given a set of genomic sequences, MEME finds overrepresented patterns using expectation-maximization.

All three are connected in this module: you will extract FUR binding site coordinates from a paper, pull the corresponding sequences from the *E. coli* K-12 genome using Biopython, and run MEME to discover the FUR binding motif.

**FUR (ferric uptake regulator)** is a well-characterized transcription factor in *E. coli* that represses iron-acquisition genes when iron is abundant. Its binding sites are a good test case because the motif (the "Fur box") is experimentally well-established and documented in the literature.

---

**Learning objectives:**
- Extract structured data from unstructured paper text using Claude
- Validate extracted data against the source
- Use Biopython to extract genomic sequences by coordinate
- Run MEME and interpret motif output (E-value, logo shape)
- Align paired-end reads with bowtie2
- Compute TSS distances as a downstream regulatory analysis

## 1. Paper → Structured Data with Claude

When papers report binding sites, gene lists, or genomic coordinates, these are often buried in supplementary tables, result paragraphs, or figure captions. Manually copying them is error-prone. Claude can extract them into a structured format (CSV/TSV) far more quickly.

**Workflow:**
1. Paste the relevant paper excerpt (or upload the PDF) to Claude.
2. Ask Claude to extract the data into a specific CSV schema (e.g., `gene, start, end, strand, score`).
3. **Always validate** what Claude extracts against the original table — spot-check at least 3 entries before using the data downstream. LLMs can mis-read numbers, especially in dense tables.

**Why validation matters:** A one-digit error in a genomic coordinate can place you in the wrong gene entirely. The validation step is not optional.

### Exercise 1 — Extract FUR binding sites from a paper excerpt

A paper excerpt describing FUR binding sites in *E. coli* is provided in `data/sample/fur_paper_excerpt.txt`.

1. Open the file and read the excerpt.
2. Use Claude to extract the binding site coordinates into a CSV with columns: `gene_name, chrom, start, end, strand`.
3. Validate **at least 3 entries** by going back to the original text and confirming the numbers match.
4. Paste the final validated CSV content in the code cell below (as a Python string or by writing it to `data/fur_binding_sites.csv`).

> **Tip:** If Claude makes an error, correct it manually and note what went wrong — this is part of developing good extraction intuition.

In [ ]:
# Paste or write your extracted CSV content here.
# Example schema: gene_name, chrom, start, end, strand
# You can write it as a Python string and save it, or write directly to a file.


## 2. Cross-Checking Against Published Regulons

Before interpreting your own results, you need to know what is already established in the literature. For well-studied organisms like *E. coli* K-12, published regulons (the set of genes controlled by a transcription factor) are available in databases such as **RegulonDB** and **EcoCyc**.

Understanding the published reference for FUR will help you:
- Assess whether your extracted coordinates are consistent with the literature.
- Identify the experimental methods used to establish the regulon (e.g., ChIP-seq, SELEX, DNase footprinting).
- Frame your downstream MEME results in biological context.

### Exercise 2 — Find the published FUR regulon reference

Use Claude to answer the following:
1. What published reference gene sets exist for the FUR regulon in *E. coli*?
2. What experimental methods were used to identify them?
3. Which databases catalog this regulon?

Write a **3-sentence summary** in the markdown cell below. Use this framing when you interpret your own MEME results in Exercise 6.

*Your 3-sentence summary here.*

## 3. How MEME Works

**MEME (Multiple EM for Motif Elicitation)** uses the expectation-maximization (EM) algorithm to find overrepresented sequence patterns across a set of input sequences. The algorithm iterates between:
- **E-step:** estimating the probability that each position in each sequence is part of the motif.
- **M-step:** updating the position weight matrix (PWM) that best explains those assignments.

**Key concepts:**

| Term | Meaning |
|------|---------|
| **E-value** | How often you would expect a motif at least this good by chance, given the database size. Low E-value (e.g., < 0.05) means the motif is unlikely to be random. |
| **Width** | Length of the motif. Controlled by `-minw` and `-maxw` flags. For transcription factor binding sites, typical range is 15–25 bp. |
| **PWM / logo** | Position weight matrix visualized as a sequence logo. Tall letters = high conservation at that position. |
| **`-mod oops`** | One occurrence per sequence — assumes every input sequence contains the motif exactly once. Use when your input is curated binding sites. |
| **`-mod zoops`** | Zero or one per sequence — more permissive. Use when you are less sure. |

**Important:** MEME takes **DNA sequences** (FASTA format), not coordinates. You must extract sequences first (Exercise 5).

### Exercise 3 — Understand MEME in your own words

Use Claude Code to explore the MEME algorithm and E-value interpretation. Then write a **3-sentence summary in your own words** — not copied from Claude's output, but re-stated in your own understanding.

> Focus on: what problem EM is solving, what E-value measures, and why motif width matters.

*Your 3-sentence summary here.*

### Exercise 4 — Conceptual prediction (answer before asking Claude)

**Without asking Claude Code first:** predict what the MEME output would look like for each of these two inputs:

(a) 10 **random** 200 bp genomic sequences from *E. coli* K-12  
(b) 10 **real FUR binding site** sequences (±50 bp around known sites)

For each case, predict: approximate E-value range, whether a clear motif logo would appear, and what width you would expect.

Write your prediction below. Then verify your reasoning with Claude Code and note where you were right or wrong.

*Your prediction here (before checking with Claude Code).*

---

*Verification notes (after checking with Claude Code):*

## 4. Extracting Sequences with Biopython

Once you have genomic coordinates, you need to extract the actual DNA sequences for MEME input. Biopython's `SeqIO` and `SeqRecord` modules make this straightforward.

**General approach:**
1. Load the reference FASTA (`SeqIO.to_dict()` or indexed with `SeqIO.index()`).
2. For each coordinate row, slice the chromosome sequence with a window (e.g., ±50 bp from center).
3. Handle strand — if strand is `-`, take the reverse complement (`seq.reverse_complement()`).
4. Write sequences to a FASTA file for MEME input.

**Reference:** *E. coli* K-12 MG1655 complete genome — NCBI accession `NC_000913.3` (FASTA can be downloaded with `efetch` or `datasets`).

```python
# Minimal example skeleton
from Bio import SeqIO

genome = SeqIO.to_dict(SeqIO.parse("ecoli_k12.fasta", "fasta"))
chrom_seq = genome["NC_000913.3"].seq

window = 50
fragment = chrom_seq[start - window : end + window]  # adjust as needed
```

### Exercise 5 — Write the sequence extraction script

Use Claude Code to generate a Biopython script that:
1. Reads the CSV of FUR binding site coordinates you produced in Exercise 1.
2. Loads the *E. coli* K-12 reference FASTA.
3. Extracts sequences ±50 bp around each binding site center.
4. Handles strand correctly (reverse complement for `-` strand entries).
5. Writes a FASTA file (`data/fur_sites_for_meme.fasta`) ready for MEME input.

Run the script and confirm the output FASTA looks correct before proceeding.

Write the final working script in the code cell below.

In [ ]:
# Your Biopython sequence extraction script here.


## 5. Running MEME

MEME is a command-line tool. The basic invocation is:

```bash
meme input.fasta -dna -oc output_dir -mod oops -minw 15 -maxw 25 -nmotifs 3
```

Use **Claude Code plan mode** to generate the full command with the flags appropriate for your FUR binding site dataset. Make sure you understand each flag before running.

In [ ]:
%%bash
# Run MEME on your extracted sequences.
# Use Claude Code with plan mode to generate the full meme command with appropriate flags.
# Annotate each flag with a comment explaining what it does.

# meme [input.fasta] [flags]


### Exercise 6 — Interpret your MEME results

After running MEME:
1. Use Claude Code to help you read the `meme.txt` or `meme.html` output.
2. Then write your **own assessment** — do not just copy Claude's interpretation.

Address all three of the following:
- **E-value:** Is the top motif statistically significant? What does the E-value tell you?
- **Motif logo:** Does the shape match what you expect for a FUR binding site (the "Fur box")? Which positions are most conserved?
- **Biological meaning:** Given your background from Exercise 2 (the published FUR regulon), is this result consistent with the literature? Why or why not?

Write your assessment in the markdown cell below.

*Your MEME result assessment here.*

## 6. Paired-End Alignment with bowtie2

For RNA-seq and ChIP-seq data, reads often come in **pairs** (R1 and R2) because sequencing is performed from both ends of each DNA fragment. Paired-end data provides:
- Higher confidence alignments (both reads must map consistently).
- Insert size information (distance between the two mapped reads).

**Key bowtie2 flags for paired-end mode:**

| Flag | Meaning |
|------|---------|
| `-1 <file>` | FASTQ file for the first reads in each pair (R1) |
| `-2 <file>` | FASTQ file for the second reads in each pair (R2) |
| `-X <int>` | Maximum insert size (default 500 bp). If the mapped distance between R1 and R2 exceeds this, the pair is discarded. Adjust based on your library preparation protocol. |
| `--no-mixed` | Only report alignments where **both** reads in a pair map. Discard pairs where only one read maps. |
| `--no-discordant` | Only report **concordant** pairs (R1 and R2 map in the expected orientation and distance). Discards pairs that map in unexpected orientations. |

**Typical paired-end command:**
```bash
bowtie2 -x genome_index -1 sample_R1.fastq.gz -2 sample_R2.fastq.gz \
    -X 500 --no-mixed --no-discordant \
    -p 4 -S output.sam
```

### Exercise 7 — Understand the paired-end flags

Use Claude Code to explore each of the three flags below. Then write a **one-sentence explanation for each** in your own words — focus on *why* you would use each flag, not just what it does mechanically.

- `-X` (maximum insert size)
- `--no-mixed`
- `--no-discordant`

Also answer: what would go wrong in a downstream analysis if you forgot `--no-discordant`?

**`-X`:** *Your one-sentence explanation.*

**`--no-mixed`:** *Your one-sentence explanation.*

**`--no-discordant`:** *Your one-sentence explanation.*

**What goes wrong without `--no-discordant`:** *Your answer.*

## 7. TSS Distance Analysis

A common downstream regulatory analysis is computing the distance between transcription factor binding sites and **transcription start sites (TSS)**. This answers the biological question: *does this factor predominantly bind near promoters, far upstream, or in gene bodies?*

For FUR, which is a repressor, we expect binding sites to cluster near or just upstream of TSS — blocking RNA polymerase access.

**Approach:**
1. You have two GFF files: one for binding sites, one for TSS positions.
2. For each binding site, find the nearest TSS and compute the distance.
3. Plot the distribution of distances.

This is the same biological question as Session 8 — here you are implementing it from scratch.

**Useful Python tools:** `pandas`, `numpy`, or `pybedtools`. You can also implement it with a simple nested loop for small datasets.

### Exercise 8 — TSS distance script

Use Claude Code to generate a Python script that:
1. Reads two GFF files: one with binding site intervals, one with TSS positions.
2. For each binding site, computes the distance to the nearest TSS (same chromosome).
3. Reports: minimum distance, median distance, and fraction of binding sites within 500 bp of a TSS.
4. (Optional) Plots the distance distribution as a histogram.

Write the final working script in the code cell below.

> **Biological framing:** Based on Exercise 2 (the published FUR regulon), what distance distribution do you expect? Write your prediction as a comment at the top of the script.

In [ ]:
# TSS distance analysis script.
# Prediction (write before running): ...



## End of Module 4

You have completed:
- Extracting structured genomic data from paper text using Claude
- Validating extracted data against source material
- Using Biopython to pull sequences from a reference genome
- Running MEME and interpreting motif output
- Understanding paired-end bowtie2 alignment flags
- Computing TSS distances as a downstream regulatory analysis

---

Run `/log` before closing this session.